# Clasificación de encabezados de noticias

**Objetivo:** adaptar el código proporcionado para clasificar encabezados de noticias en las categorías `b`, `e`, `m` y `t` usando **Doc2Vec + SVM**.

Categorías:
- `b`: Business
- `e`: Entertainment
- `m`: Health
- `t`: Science/Technology

El archivo contiene 422,937 registros. Para hacer la ejecución reproducible en un entorno limitado, se utiliza una muestra estratificada de 50,000 encabezados después de cargar y revisar el CSV completo.

In [ ]:
# Librerías
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Intentar descargar recursos NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Cargar CSV
data = pd.read_csv("newsCorpora-trimmed.csv")

print("Dimensiones originales:", data.shape)
print(data.head())
print(data.isna().sum())

Dimensiones originales: (422937, 2)
  category                                               text
0        b  Fed official says weak data caused by weather,...
1        b  Fed's Charles Plosser sees high bar for change...
2        b  US open: Stocks fall after Fed official hints ...
3        b  Fed risks falling 'behind the curve', Charles ...
4        b  Fed's Plosser: Nasty Weather Has Curbed Job Gr...


In [ ]:
# Limpieza y selección de categorías
data = data.dropna(subset=["category"]).copy()
categories = ['b', 't', 'e', 'm']
data = data[data["category"].isin(categories)]

print("Distribución completa:")
print(data["category"].value_counts())

# Muestra estratificada para una ejecución eficiente
data, _ = train_test_split(
    data, train_size=50000, stratify=data["category"], random_state=42
)
data = data.reset_index(drop=True)

print("\nMuestra utilizada:", data.shape)
print(data["category"].value_counts())

Distribución completa:
e    152821
b    115967
t    108501
m     45639
Name: count, dtype: int64

Muestra utilizada: (50000, 2)
e    18070
b    13710
t    12825
m     5395
Name: count, dtype: int64


In [ ]:
# Preprocesamiento de texto
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    # Respaldo si NLTK no tiene descargado el corpus localmente
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    stop_words = set(ENGLISH_STOP_WORDS)

def preprocess_text(text):
    # Tokenización robusta para encabezados
    tokens = re.findall(r"[A-Za-z]+", str(text).lower())
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

data["tokens"] = data["text"].apply(preprocess_text)

print(data[["text", "tokens", "category"]].head())

In [ ]:
# Crear documentos etiquetados
tagged_data = [
    TaggedDocument(words=row["tokens"], tags=[str(i)])
    for i, row in data.iterrows()
]

# Entrenar Doc2Vec
model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    dm=0,
    dbow_words=1,
    seed=42
)

model.build_vocab(tagged_data)
model.train(
    tagged_data,
    total_examples=model.corpus_count,
    epochs=model.epochs
)

print("Tamaño del vocabulario:", len(model.wv))

In [ ]:
# Obtener vectores de documentos
X = np.array([model.dv[str(i)] for i in range(len(data))])
y = data["category"].values

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
# Clasificador SVM
classifier = SVC(kernel="linear")
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Accuracy: 0.8388 (83.88%)

Reporte de clasificación:
              precision    recall  f1-score   support

           b     0.8025    0.8180    0.8102      2742
           e     0.8877    0.9272    0.9070      3614
           m     0.8351    0.7275    0.7776      1079
           t     0.8068    0.7832    0.7949      2565

    accuracy                         0.8388     10000
   macro avg     0.8330    0.8140    0.8224     10000
weighted avg     0.8379    0.8388    0.8377     10000


In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title("Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.xticks(range(4), ["b","e","m","t"])
plt.yticks(range(4), ["b","e","m","t"])
for i in range(4):
    for j in range(4):
        plt.text(j, i, str(cm[i,j]), ha="center", va="center")
plt.tight_layout()
plt.show()

[[2243  144   63  292]
 [ 110 3351   36  117]
 [ 124   98  785   72]
 [ 318  182   56 2009]]

In [ ]:
# Función para clasificar un nuevo encabezado
def classify_new_document(text):
    tokens = preprocess_text(text)
    vector = model.infer_vector(tokens)
    return classifier.predict([vector])[0]

examples = [
    "New breakthrough in cancer research",
    "Stocks rise after strong market report",
    "Actor announces new movie project",
    "Scientists develop new technology"
]

for headline in examples:
    print(headline, "->", classify_new_document(headline))